# Add noise to a Stim circuit

A qLDPC noise model adds noise to a `stim.Circuit` by inserting error instructions, so the same circuit can be studied under different noise assumptions.
This notebook adds noise to a small circuit in increasingly customized ways, and then applies a hardware-inspired noise model to a surface-code memory experiment.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import math

import stim

from qldpc import circuits, codes

## A small circuit

A three-qubit GHZ-state preparation, small enough that the noise added to it is easy to read:

In [3]:
circuit = stim.Circuit("""
    R 0 1 2
    H 0
    CX 0 1 0 2
    M 0 1 2
""")
print(circuit)

R 0 1 2
H 0
CX 0 1 0 2
M 0 1 2


## Uniform depolarizing noise

`DepolarizingNoiseModel(p)` gives every operation noise with probability `p`.
A one-qubit Clifford is followed by a uniformly random one-qubit Pauli in `{X,Y,Z}` with total probability `p`, and a two-qubit Clifford by a uniformly random Pauli string from `{I,X,Y,Z}^2 \ {II}` with total probability `p`.
Measurements report the wrong outcome, and resets prepare the orthogonal state, with probability `p`.
Idle noise is off unless `include_idling_error=True`:

In [4]:
p = 1e-3
no_idle_noise = circuits.DepolarizingNoiseModel(p)
with_idle_noise = circuits.DepolarizingNoiseModel(p, include_idling_error=True)

print("without idle noise:")
print(no_idle_noise.noisy_circuit(circuit))
print()
print("with idle noise:")
print(with_idle_noise.noisy_circuit(circuit))

without idle noise:
R 0 1 2
X_ERROR(0.001) 0 1 2
H 0
DEPOLARIZE1(0.001) 0
CX 0 1
DEPOLARIZE2(0.001) 0 1
CX 0 2
DEPOLARIZE2(0.001) 0 2
M(0.001) 0 1 2

with idle noise:
DEPOLARIZE1(0.001) 2
R 0 1 2
X_ERROR(0.001) 0 1 2
H 0
DEPOLARIZE1(0.001) 0
CX 0 1
DEPOLARIZE2(0.001) 0 1
CX 0 2
DEPOLARIZE2(0.001) 0 2
M(0.001) 0 1 2
DEPOLARIZE1(0.001) 1


## Set operation-specific error rates

Use `NoiseModel` when one- and two-qubit gates, measurements, resets, and idle moments should have independent rates.
A float rate means uniform depolarizing noise for gates and idle moments, and a bit-flip probability for measurements and resets.

In [5]:
custom_noise = circuits.NoiseModel(
    clifford_1q_error=0.002,
    clifford_2q_error=0.02,
    readout_error=0.05,
    reset_error=0.01,
    idle_error=0.0001,
)
print(custom_noise.noisy_circuit(circuit))

DEPOLARIZE1(0.0001) 2
R 0 1 2
X_ERROR(0.01) 0 1 2
H 0
DEPOLARIZE1(0.002) 0
CX 0 1
DEPOLARIZE2(0.02) 0 1
CX 0 2
DEPOLARIZE2(0.02) 0 2
M(0.05) 0 1 2
DEPOLARIZE1(0.0001) 1


## Pauli channels

A dictionary from Pauli strings to probabilities defines any Pauli channel, such as biased noise on dephasing-dominated qubits.
In a two-qubit string, the first character acts on a gate's first target and the second on its second target:

In [6]:
biased_noise = circuits.NoiseModel(
    clifford_1q_error={"Z": 0.002, "X": 0.0002, "Y": 0.0002},
    clifford_2q_error={"ZI": 0.004, "IZ": 0.004, "ZZ": 0.002},
    idle_error={"Z": 0.001},
)
print(biased_noise.noisy_circuit(circuit))

Z_ERROR(0.001) 2
R 0 1 2
H 0
PAULI_CHANNEL_1(0.0002, 0.0002, 0.002) 0
CX 0 1
PAULI_CHANNEL_2(0, 0, 0.004, 0, 0, 0, 0, 0, 0, 0, 0, 0.004, 0, 0, 0.002) 0 1
CX 0 2
PAULI_CHANNEL_2(0, 0, 0.004, 0, 0, 0, 0, 0, 0, 0, 0, 0.004, 0, 0, 0.002) 0 2
M 0 1 2
Z_ERROR(0.001) 1


The strings of a `PauliChannel`, or of a dictionary, are mutually exclusive errors, while a `PauliChannelSequence` lists independent errors, any combination of which may occur.
Gates on three or more qubits, such as `SPP` Pauli-product rotations, take their noise from `clifford_nq_error`, keyed by their number of qubits:

In [7]:
two_qubit_gate = stim.Circuit("CZ 0 1")
exclusive = circuits.PauliChannel({"XX": 0.01, "ZZ": 0.01})
independent = circuits.PauliChannelSequence({"XX": 0.01, "ZZ": 0.01})

print("mutually exclusive errors:")
print(circuits.NoiseModel(clifford_2q_error=exclusive).noisy_circuit(two_qubit_gate))
print()
print("independent errors:")
print(circuits.NoiseModel(clifford_2q_error=independent).noisy_circuit(two_qubit_gate))
print()

three_qubit_noise = circuits.NoiseModel(
    clifford_nq_error={3: circuits.PauliChannel({"XXX": 0.01, "ZZZ": 0.02})}
)
print("three-qubit Pauli-product rotation:")
print(three_qubit_noise.noisy_circuit(stim.Circuit("SPP X0*X1*X2")))

mutually exclusive errors:
CZ 0 1
PAULI_CHANNEL_2(0, 0, 0, 0, 0.01, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.01) 0 1

independent errors:
CZ 0 1
E(0.01) X0 X1
E(0.01) Z0 Z1

three-qubit Pauli-product rotation:
SPP X0*X1*X2
E(0.01) X0 X1 X2
ELSE_CORRELATED_ERROR(0.020202) Z0 Z1 Z2


## Scheduling, moments, and idle noise

Idle noise depends on how long qubits wait, which a circuit does not fully specify.
A noise model therefore uses its `scheduler` to place the operations of a circuit into moments: units of time in which each qubit is either busy or idle.
The default scheduler, `ASAPScheduler()`, ignores `TICK`s in the input circuit, places each gate as early as possible and then each reset as late as the other gates allow, and lets single-qubit gates take no time.
`ALAPScheduler()` places gates as late as possible instead, and `TickScheduler()` takes the moments of a circuit from its `TICK`s; call `help` on a scheduler to see its options.
`TICK`s can set the timing of circuits in two ways.
With `scheduler=TickScheduler()`, the `TICK`s of each circuit set its moments.
With `scheduler=TickScheduler().get_schedule(timed_circuit)`, a noise model takes the moments of each circuit from the schedule of `timed_circuit`.
Such a circuit may order operations differently from `timed_circuit`.
It must otherwise match `timed_circuit`, as `help(circuits.noise_model)` describes.

A noise model with a scheduler schedules each circuit implicitly.
To inspect that schedule, call `noise_model.scheduler.get_schedule(circuit)`:

In [8]:
schedule = with_idle_noise.scheduler.get_schedule(circuit)
for moment, operations in enumerate(schedule.get_moments()):
    print(f"moment {moment}:", *operations)

moment 0: R 0 R 1 H 0
moment 1: R 2 CX 0 1
moment 2: CX 0 2 M 1
moment 3: M 0 M 2


Qubit 2 is reset just before its first gate, and qubit 1 is measured as soon as its gates end, so each idles for one moment, which gives the two `DEPOLARIZE1(0.001)` errors in the noisy circuit above.

Slow noise, such as a qubit frequency that drifts from one shot to the next, makes idle errors grow quadratically in time: a `quadratic_idle_error` with probability $p$ per moment becomes about $k^2 p$ over $k$ consecutive idle moments.
Below, `TickScheduler()` gives each Hadamard gate a moment, so qubits 0 and 2 each idle for two consecutive moments, and qubit 1 for one moment on each side of its gate.

In [9]:
timed_circuit = stim.Circuit("""
    H 0
    TICK
    H 1
    TICK
    H 2
""")
quadratic_idle_noise = circuits.NoiseModel(
    idle_error={"Z": 0.001},
    quadratic_idle_error={"Z": 0.002},
    scheduler=circuits.TickScheduler(),
)
print(quadratic_idle_noise.noisy_circuit(timed_circuit))

H 0
TICK
Z_ERROR(0.002996) 1
H 1
TICK
Z_ERROR(0.00991835) 2
H 2
Z_ERROR(0.00991835) 0
Z_ERROR(0.002996) 1


Each idle run gets one combined error: about $0.001 + 0.002$ for one moment, and about $2 \times 0.001 + 4 \times 0.002 = 0.01$ for two.
These settings model idling in the circuit as written; compilation and dynamical decoupling on hardware change where and how qubits idle, so idle-noise parameters depend on how circuits are compiled and run.

## Override noise for specific gates

`rules` assigns a `NoiseRule` to a gate by name, overriding the default for gates of its size.
A rule can add a Pauli channel after the gate, change the error of a measurement or reset, or add a fragment of Stim noise instructions, written for qubits `0, 1, ...` and applied to the targets of each gate:

In [10]:
gate_specific_noise = circuits.NoiseModel(
    clifford_1q_error=0.001,
    clifford_2q_error=0.01,
    readout_error=0.01,
    rules={
        "CZ": circuits.NoiseRule(after={"ZZ": 0.02}),
        "MR": circuits.NoiseRule(readout_error=0.03),
        "SQRT_X": circuits.NoiseRule(after=stim.Circuit("X_ERROR(0.004) 0\nZ_ERROR(0.001) 0")),
    },
)
gate_circuit = stim.Circuit("""
    R 0 1 2
    SQRT_X 0
    H 1
    CZ 0 1
    CX 1 2
    MR 0
    M 1 2
""")
print(gate_specific_noise.noisy_circuit(gate_circuit))

R 0 1 2
SQRT_X 0
H 1
X_ERROR(0.004) 0
Z_ERROR(0.001) 0
DEPOLARIZE1(0.001) 1
CZ 0 1
PAULI_CHANNEL_2(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.02) 0 1
CX 1 2
MR(0.03) 0
DEPOLARIZE2(0.01) 1 2
M(0.01) 1 2


## Advanced noise rules

`rule_func` is a callback that receives each gate application, with its targets and tag, and returns a `NoiseRule`, or `None` to fall back to the other rules, which it takes precedence over.
For example, a circuit can mark idle periods with their durations, as in `I[idle_us=5] 0`, and a `rule_func` can turn them into Z errors with probability $(1 - e^{-at - bt^2})/2$ for an idle of $t$ microseconds:

In [11]:
a = 0.002
b = 0.0001


def duration_dependent_idle_rule(
    op: stim.CircuitInstruction,
) -> circuits.NoiseRule | None:
    if op.name != "I" or not op.tag.startswith("idle_us="):
        return None

    duration_us = float(op.tag.removeprefix("idle_us="))
    exponent = a * duration_us + b * duration_us**2
    error_probability = (1 - math.exp(-exponent)) / 2
    return circuits.NoiseRule(after=circuits.PauliChannel({"Z": error_probability}))


idle_circuit = stim.Circuit("""
    I[idle_us=5] 0
    I[idle_us=20] 1
""")
duration_dependent_noise = circuits.NoiseModel(rule_func=duration_dependent_idle_rule)
print(duration_dependent_noise.noisy_circuit(idle_circuit))

I[idle_us=5] 0
I[idle_us=20] 1
Z_ERROR(0.0062111) 0
Z_ERROR(0.0384418) 1


## Protect qubits and operations from noise

Parts of a circuit can be kept noiseless, such as a bookkeeping ancilla or an ideal state preparation:

- Qubits passed in `immune_qubits`, or whose `QUBIT_COORDS` are tagged with `circuits.DEFAULT_IMMUNE_QUBIT_TAG`, are immune to noise.
- Instructions and `REPEAT` blocks whose tag contains `circuits.DEFAULT_IMMUNE_OP_TAG` are noiseless, and `circuits.as_noiseless_circuit` wraps a whole circuit in such a block.

A gate on an immune qubit is noiseless, or, with `immunize_gates=False`, keeps only the part of its noise that acts trivially on every immune qubit:

In [12]:
immunity_circuit = stim.Circuit(f"""
    QUBIT_COORDS[{circuits.DEFAULT_IMMUNE_QUBIT_TAG}](0, 0) 0
    R 0 1 2
    H[{circuits.DEFAULT_IMMUNE_OP_TAG}] 1
    CX 1 2
    CX 0 1
""")
gate_noise = circuits.DepolarizingNoiseModel(p)

print("immune gates:")
print(gate_noise.noisy_circuit(immunity_circuit))
print()
print("noise conditioned on immune qubits:")
print(gate_noise.noisy_circuit(immunity_circuit, immunize_gates=False))
print()
print("a noiseless preparation followed by a noisy measurement:")
noiseless_prep = circuits.as_noiseless_circuit(stim.Circuit("R 0 1\nH 0\nCX 0 1"))
print(gate_noise.noisy_circuit(noiseless_prep + stim.Circuit("M 0 1")))

immune gates:
QUBIT_COORDS[__IMMUNE_QUBIT_TO_NOISE__](0, 0) 0
R 0 1 2
X_ERROR(0.001) 1 2
H[__IMMUNE_TO_NOISE__] 1
CX 1 2
DEPOLARIZE2(0.001) 1 2
CX 0 1

noise conditioned on immune qubits:
QUBIT_COORDS[__IMMUNE_QUBIT_TO_NOISE__](0, 0) 0
R 0 1 2
X_ERROR(0.001) 1 2
H[__IMMUNE_TO_NOISE__] 1
CX 1 2
DEPOLARIZE2(0.001) 1 2
CX 0 1
DEPOLARIZE1(0.0002) 1

a noiseless preparation followed by a noisy measurement:
REPEAT[__IMMUNE_TO_NOISE__] 1 {
    R 0 1
    H 0
    CX 0 1
}
M(0.001) 0 1


## A noisy surface-code memory experiment

`get_memory_experiment` prepares a logical state, measures stabilizers for several rounds, and reads out the logical state, and with `noise_model=` it returns the noisy circuit.
Here the noise model is `SI1000NoiseModel`, from [A Fault-Tolerant Honeycomb Memory](https://arxiv.org/abs/2108.10457), which keeps the `TICK` layers of a circuit as its moments:

In [13]:
code = codes.SurfaceCode(3)
num_rounds = 3
memory_noise = circuits.SI1000NoiseModel(0.01)
noisy_memory = circuits.get_memory_experiment(
    code,
    basis="Z",
    num_rounds=num_rounds,
    noise_model=memory_noise,
)

print("code parameters:", code.get_code_params())
print("circuit qubits:", noisy_memory.num_qubits)
print("measurements:", noisy_memory.num_measurements)
print("detectors:", noisy_memory.num_detectors)
print("observables:", noisy_memory.num_observables)

code parameters: (9, 1, 3)
circuit qubits: 17
measurements: 33
detectors: 16
observables: 1


Syndromes are the stabilizer-measurement outcomes of each round; this Z-basis experiment builds its detectors from the Z-type checks shown here:

In [14]:
measurement_sampler = noisy_memory.compile_sampler(seed=1234)
measurements = measurement_sampler.sample(shots=1)[0]

syndromes = measurements[: -len(code)].reshape(num_rounds, 2, code.num_checks // 2)
syndromes_z = syndromes[:, 1, :]


def bitmask(bits):
    return "".join("1" if bit else "_" for bit in bits)


print("Z-check outcomes by round:")
for syndrome in syndromes_z:
    print(bitmask(syndrome))

Z-check outcomes by round:
_1__
____
____


Detector events flag changes in a check's outcome between rounds, and between the first round and the prepared state or the last round and the final readout, so there is one more time slice than rounds:

In [15]:
detector_sampler = noisy_memory.compile_detector_sampler(seed=1234)
detectors = detector_sampler.sample(shots=1)[0]
detectors_by_time = detectors.reshape(num_rounds + 1, code.num_checks_z)

print("detector events by time slice:")
for events in detectors_by_time:
    print(bitmask(events))

detector events by time slice:
_1__
_1__
____
____


The [logical-error-rate notebooks](logical_error_rates/2_quantum_memory_x_or_z.ipynb) take the next step: they hand noisy memory circuits to Sinter, configure a decoder, and estimate failure rates over many shots.